# Interior Design Chatbot — Version 2: QLoRA Fine-Tuning

Fine-tune **Qwen2.5-1.5B-Instruct** on an interior design dataset using **QLoRA** (4-bit quantisation + LoRA adapters).

---
| Component | Choice |
|-----------|--------|
| Base model | `Qwen/Qwen2.5-1.5B-Instruct` |
| Method | QLoRA (4-bit + LoRA) |
| Dataset | `bitext/Bitext-interior-design-llm-chatbot-training-dataset` (HuggingFace) |
| Libraries | `transformers`, `peft`, `trl`, `bitsandbytes`, `datasets` |

**Colab setup:** Runtime → Change runtime type → **T4 GPU** (15 GB VRAM)

In [15]:
!zip -r qwen-interior-design-qlora.zip qwen-interior-design-qlora/

  adding: qwen-interior-design-qlora/ (stored 0%)
  adding: qwen-interior-design-qlora/README.md (deflated 43%)
  adding: qwen-interior-design-qlora/checkpoint-5/ (stored 0%)
  adding: qwen-interior-design-qlora/checkpoint-5/trainer_state.json (deflated 59%)
  adding: qwen-interior-design-qlora/checkpoint-5/scheduler.pt (deflated 62%)
  adding: qwen-interior-design-qlora/checkpoint-5/optimizer.pt (deflated 10%)
  adding: qwen-interior-design-qlora/checkpoint-5/training_args.bin (deflated 53%)
  adding: qwen-interior-design-qlora/checkpoint-5/README.md (deflated 65%)
  adding: qwen-interior-design-qlora/checkpoint-5/tokenizer.json (deflated 81%)
  adding: qwen-interior-design-qlora/checkpoint-5/chat_template.jinja (deflated 71%)
  adding: qwen-interior-design-qlora/checkpoint-5/tokenizer_config.json (deflated 60%)
  adding: qwen-interior-design-qlora/checkpoint-5/adapter_model.safetensors (deflated 29%)
  adding: qwen-interior-design-qlora/checkpoint-5/adapter_config.json (deflated 58%)

## Section 1 — Install Dependencies

In [2]:
!pip install transformers peft trl bitsandbytes datasets accelerate ipywidgets -q
!pip install pyarrow --upgrade -q
print("✓ All dependencies installed — now RESTART RUNTIME before continuing")
print("✓ All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.4 MB/s eta 0:00:00
✓ All dependencies installed — now RESTART RUNTIME before continuing
✓ All dependencies installed


## Section 2 — Imports & GPU Check

In [3]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import ipywidgets as widgets
from datasets import Dataset
from IPython.display import display, HTML, clear_output
from trl import SFTTrainer, SFTConfig

if torch.cuda.is_available():
    print(f"✓ GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("✗ No GPU — go to Runtime → Change runtime type → T4 GPU")

✓ GPU  : Tesla T4
  VRAM : 15.6 GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 3 — Load & Prepare Dataset

In [5]:
DATASET_NAME = "disham993/Synthetic_Furniture_Dataset"

print(f"Loading dataset: {DATASET_NAME}")
raw = load_dataset(DATASET_NAME, split="train")
print(f"Raw size : {len(raw)} rows")
print(f"Columns  : {raw.column_names}")
print()
print("Sample row:")
print(raw[0])

# Columns: name, description, ad
# Convert into instruction / response pairs focused on material & texture
def make_qa(sample):
    instruction = (
        f"I am looking for a {sample['name']}. "
        f"What materials and textures should I look for, and how should it look and feel?"
    )
    response = (
        f"{sample['description']} {sample['ad']}"
    ).strip()
    return {"instruction": instruction, "response": response}

raw_dataset = raw.map(make_qa, remove_columns=raw.column_names)

print(f"\nDataset size after format : {len(raw_dataset)} samples")
print(f"Columns                   : {raw_dataset.column_names}")
print()
print("Formatted sample:")
print("Q:", raw_dataset[0]["instruction"])
print("A:", raw_dataset[0]["response"])

from datasets import Dataset

qa_pairs = [
    # Colour
    {"instruction": "What colors work best in a small living room?",
     "response": "Light, neutral tones like soft white, cream, or pale grey make small rooms feel larger by reflecting light. Add depth with a single accent wall in a muted sage or dusty blue. Avoid dark colours on all four walls as they absorb light and shrink the perceived space."},
    {"instruction": "How do I choose a colour palette for an open-plan space?",
     "response": "Use a consistent base colour (e.g. warm white or greige) across all zones to create flow. Differentiate areas with accent colours — a muted teal for the dining zone, soft terracotta for the lounge. Limit your palette to 3 colours: one dominant, one secondary, one accent."},
    {"instruction": "What paint colours make a bedroom feel calm and relaxing?",
     "response": "Soft blues, muted greens, and warm lavenders promote relaxation. Benjamin Moore's 'Tranquility' or Farrow & Ball's 'Mizzle' are popular choices. Avoid bright or saturated colours which stimulate rather than soothe."},
    {"instruction": "How do I use dark colours without making a room feel gloomy?",
     "response": "Balance dark walls with plenty of natural light, light-coloured furniture, and mirrors. Use dark colours on a single feature wall rather than all four. Incorporate warm lighting (2700K bulbs) to make dark tones feel cosy rather than oppressive."},
    # Furniture
    {"instruction": "How should I arrange furniture in an open-plan living and dining area?",
     "response": "Define each zone using a rug to anchor furniture groupings. Place the sofa with its back towards the dining area to create a natural boundary. Ensure clear traffic paths of at least 90 cm between pieces. Keep the dining table proportionate to the space — a 6-seater in a small open plan overwhelms."},
    {"instruction": "What furniture works best in a small bedroom?",
     "response": "Choose a bed with built-in storage drawers underneath. Use a wall-mounted bedside table instead of a floor-standing one to free up floor space. A slim-profile wardrobe with mirrored doors doubles as a mirror and creates depth. Avoid oversized dressers — use vertical shelving instead."},
    {"instruction": "How do I make a studio apartment feel like separate rooms?",
     "response": "Use a bookshelf or open shelving unit as a room divider between sleeping and living areas. A sofa placed perpendicular to the sleeping zone also acts as a visual boundary. Distinct rugs in each zone signal different functions without needing walls."},
    {"instruction": "What is the ideal sofa size for a living room?",
     "response": "Allow at least 45 cm of walking space around the sofa. For a typical living room (4×5 m), a 2.2–2.5 m three-seater is proportionate. The sofa should not exceed two-thirds of the wall it faces. Always measure doorways before purchasing."},
    # Lighting
    {"instruction": "What lighting should I use for a home office?",
     "response": "Layer three types: ambient (overhead), task (desk lamp at 450–500 lux), and accent. Position your desk lamp to the left if right-handed to avoid shadow. Use daylight bulbs (5000–6500K) to reduce eye strain during work. Avoid placing your monitor directly in front of a window to prevent glare."},
    {"instruction": "How do I create a cosy atmosphere with lighting?",
     "response": "Use warm-white bulbs (2700–3000K) and layer sources at different heights — floor lamps, table lamps, and candles all lower than eye level. Dimmer switches give you flexibility from bright task lighting to soft evening ambience. Avoid a single overhead ceiling light as the sole source."},
    {"instruction": "What is the best lighting for a kitchen?",
     "response": "Kitchens need strong task lighting (at least 500 lux) over worktops — under-cabinet LED strips work best. Add pendant lights over an island for ambience and overhead recessed lights for general illumination. Use separate circuits so you can dim ambient lights while keeping task lighting bright."},
    {"instruction": "How do I light a dark hallway?",
     "response": "Install a series of wall sconces rather than a single overhead light to distribute light evenly. A mirror opposite a light source doubles brightness. Use light paint colours and keep the floor clutter-free to maximise perceived space. Motion-sensor nightlights work well for late-night safety."},
    # Styles
    {"instruction": "What are the key principles of Scandinavian interior design?",
     "response": "Scandinavian design emphasises simplicity, functionality, and connection to nature. Key principles: (1) neutral palette of whites, greys, and natural wood tones; (2) clutter-free spaces with ample storage; (3) natural materials like oak, linen, and wool; (4) abundant natural light with sheer curtains; (5) a few carefully chosen statement pieces rather than many decorative items."},
    {"instruction": "How do I achieve an industrial interior design look?",
     "response": "Expose structural elements — brick walls, concrete floors, and steel beams. Use a palette of charcoal, black, and warm browns. Choose furniture with metal frames and reclaimed wood surfaces. Edison bulb pendant lights are a signature industrial element. Pipe shelving and raw finishes complete the look."},
    {"instruction": "What defines a minimalist interior design style?",
     "response": "Minimalism is guided by 'less is more': a limited colour palette (usually white, black, and one neutral), furniture with clean lines and no ornamentation, ample negative space, and ruthless editing of possessions. Every object should be either functional or genuinely beautiful. Concealed storage is essential."},
    {"instruction": "How do I create a bohemian interior design look?",
     "response": "Layer textiles — macramé wall hangings, patterned rugs, and mixed-print cushions. Mix furniture from different eras and origins. Incorporate abundant indoor plants. Use a warm, earthy palette with jewel-tone accents. Embrace organised clutter: eclectic collections of books, art, and travel objects tell a personal story."},
    {"instruction": "What is japandi interior design?",
     "response": "Japandi blends Japanese wabi-sabi (beauty in imperfection) with Scandinavian hygge (cosiness). Characteristics: a muted palette of warm greys, beiges, and sage greens; natural materials like bamboo, linen, and clay; low-profile furniture with clean lines; minimal decoration; and a focus on craftsmanship over mass production."},
    # Space planning
    {"instruction": "How do I make a small bathroom feel larger?",
     "response": "Use large-format floor tiles (60×60 cm or bigger) to reduce grout lines and create a seamless look. Install a frameless glass shower screen instead of a curtain. Mount the vanity on the wall to expose floor space. A large mirror above the basin visually doubles the room. Keep colours light and consistent."},
    {"instruction": "What is the rule of thirds in interior design?",
     "response": "Borrowed from photography, the rule of thirds suggests dividing a wall or room into thirds horizontally and vertically, then placing focal points at intersections. In practice: hang art at eye level (two-thirds up the wall), group décor in odd numbers, and avoid centring everything symmetrically for a more dynamic, natural composition."},
    {"instruction": "How do I add storage without making a room feel cluttered?",
     "response": "Use furniture with hidden storage: ottomans with lids, beds with drawers, and coffee tables with shelves. Install floor-to-ceiling built-in shelving — it draws the eye upward and maximises unused vertical space. Use baskets and boxes on open shelves to conceal items while keeping things visually tidy."},
    {"instruction": "How do I choose the right rug size for a living room?",
     "response": "All major furniture legs should ideally sit on the rug. For a typical sofa-and-chairs grouping, a 2.4×3 m rug is a common starting point. At minimum, the front two legs of every seat should rest on the rug. Avoid a rug that is too small — it makes the room look unanchored and smaller."},
    # Materials
    {"instruction": "What are the best flooring options for a family home?",
     "response": "Luxury vinyl plank (LVP) is highly durable, waterproof, and scratch-resistant — ideal for high-traffic areas and homes with children or pets. Engineered hardwood offers the warmth of real wood with better moisture resistance than solid hardwood. Porcelain tiles work well in kitchens and bathrooms. Avoid solid hardwood in humid areas or below grade."},
    {"instruction": "How do I choose curtains for a living room?",
     "response": "Hang curtain rods 15–20 cm above the window frame and extend them 15–20 cm beyond each side — this makes windows appear larger. Floor-length curtains (pooling 1–2 cm on the floor) look more luxurious than short ones. Choose linen or velvet for texture; sheer underlay allows privacy while admitting daylight."},
    {"instruction": "What indoor plants work best in low-light interiors?",
     "response": "Snake plants (Sansevieria), pothos, ZZ plants, and peace lilies thrive in low-light conditions. Chinese evergreens tolerate shade and add colour with patterned foliage. Cast iron plants live up to their name — nearly indestructible in dim rooms. Avoid succulents and most flowering plants in low-light spaces."},
    {"instruction": "How do I mix patterns without the room looking chaotic?",
     "response": "Follow the 60-30-10 rule: one dominant pattern (60%), one secondary pattern (30%), one accent pattern (10%). Vary scale — pair a large-scale floral with a small geometric and a subtle texture. Unify patterns through a shared colour. Keep solid-coloured furniture as a visual resting point between patterned textiles."},
    # Budget tips
    {"instruction": "How can I refresh a room on a tight budget?",
     "response": "Paint is the highest-impact, lowest-cost change you can make. Rearrange existing furniture for a fresh layout. Swap hardware on cabinets and doors. Add new cushion covers and a throw. Introduce plants — even one large fiddle-leaf fig creates drama. Declutter ruthlessly — empty space is itself a design choice."},
    {"instruction": "What are the best budget-friendly ways to add texture to a room?",
     "response": "Layer different textiles: a chunky knit throw, linen cushions, and a jute rug add warmth and texture without major expense. Peel-and-stick wallpaper on a single feature wall is affordable and removable. Baskets, woven lampshades, and macramé wall hangings introduce organic texture cheaply."},
]

dataset = Dataset.from_list(qa_pairs)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset_raw = dataset["train"]
eval_dataset_raw  = dataset["test"]

print(f"Train: {len(train_dataset_raw)} samples")
print(f"Eval : {len(eval_dataset_raw)} samples")
print("\nSample:")
print(train_dataset_raw[0])


Loading dataset: disham993/Synthetic_Furniture_Dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/432 [00:00<?, ?B/s]

data/train-00000-of-00001-654a257d4fbf9b(…):   0%|          | 0.00/76.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1003 [00:00<?, ? examples/s]

Raw size : 1003 rows
Columns  : ['name', 'description', 'ad']

Sample row:
{'name': 'Sofa', 'description': 'A comfortable seating option for your living room', 'ad': 'Sit back, relax, and sink into pure bliss with our luxurious sofa'}


Map:   0%|          | 0/1003 [00:00<?, ? examples/s]


Dataset size after format : 1003 samples
Columns                   : ['instruction', 'response']

Formatted sample:
Q: I am looking for a Sofa. What materials and textures should I look for, and how should it look and feel?
A: A comfortable seating option for your living room Sit back, relax, and sink into pure bliss with our luxurious sofa
Train: 24 samples
Eval : 3 samples

Sample:
{'instruction': 'What are the best flooring options for a family home?', 'response': 'Luxury vinyl plank (LVP) is highly durable, waterproof, and scratch-resistant — ideal for high-traffic areas and homes with children or pets. Engineered hardwood offers the warmth of real wood with better moisture resistance than solid hardwood. Porcelain tiles work well in kitchens and bathrooms. Avoid solid hardwood in humid areas or below grade.'}


In [6]:
# Format each row into Qwen chat format
# The Bitext dataset has: instruction, response (and optionally category, intent)

SYSTEM_PROMPT = (
    "You are a knowledgeable and friendly interior design consultant. "
    "Give practical, actionable advice about space planning, colour palettes, "
    "lighting, furniture arrangement, and design styles."
)

def format_sample(sample):
    """Convert a dataset row to Qwen chat format string."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": sample["instruction"]},
        {"role": "assistant", "content": sample["response"]},
    ]
    return {"text": tokenizer_ref.apply_chat_template(messages, tokenize=False)}

# We need the tokenizer to apply the chat template — load it briefly here
from transformers import AutoTokenizer as _AT
tokenizer_ref = _AT.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

dataset = raw_dataset.map(format_sample, remove_columns=raw_dataset.column_names)

# Train / validation split
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"Train samples : {len(train_dataset)}")
print(f"Eval  samples : {len(eval_dataset)}")
print()
print("Formatted sample (truncated):")
print(train_dataset[0]["text"][:500], "...")

from transformers import AutoTokenizer as _AT
tokenizer_ref = _AT.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM_PROMPT = (
    "You are a knowledgeable and friendly interior design consultant. "
    "Give practical, actionable advice about space planning, colour palettes, "
    "lighting, furniture arrangement, and design styles."
)

def format_sample(sample):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": sample["instruction"]},
        {"role": "assistant", "content": sample["response"]},
    ]
    return {"text": tokenizer_ref.apply_chat_template(messages, tokenize=False)}

train_dataset = train_dataset_raw.map(format_sample, remove_columns=["instruction", "response"])
eval_dataset  = eval_dataset_raw.map(format_sample,  remove_columns=["instruction", "response"])

print(f"Formatted. Sample (truncated):\n{train_dataset[0]['text'][:400]}...")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1003 [00:00<?, ? examples/s]

Train samples : 952
Eval  samples : 51

Formatted sample (truncated):
<|im_start|>system
You are a knowledgeable and friendly interior design consultant. Give practical, actionable advice about space planning, colour palettes, lighting, furniture arrangement, and design styles.<|im_end|>
<|im_start|>user
I am looking for a Chaise Lounge. What materials and textures should I look for, and how should it look and feel?<|im_end|>
<|im_start|>assistant
A luxurious and comfortable lounge chair for relaxation and style. Indulge in ultimate comfort and style with our eleg ...


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Formatted. Sample (truncated):
<|im_start|>system
You are a knowledgeable and friendly interior design consultant. Give practical, actionable advice about space planning, colour palettes, lighting, furniture arrangement, and design styles.<|im_end|>
<|im_start|>user
What are the best flooring options for a family home?<|im_end|>
<|im_start|>assistant
Luxury vinyl plank (LVP) is highly durable, waterproof, and scratch-resistant ...


## Section 4 — Load Base Model in 4-bit (QLoRA)

In [7]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./qwen-interior-design-qlora"

# 4-bit quantisation config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best quality for QLoRA
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,      # nested quantisation saves ~0.4 GB
)

print(f"Loading tokenizer : {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token   # Qwen has no pad token by default
tokenizer.padding_side = "right"            # required for SFT packing

print(f"Loading model (4-bit) : {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False              # disable KV-cache during training

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ Model loaded | VRAM used: {used:.1f} / {total:.1f} GB")
else:
    print("✓ Model loaded on CPU — switch to T4 GPU for training")

Loading tokenizer : Qwen/Qwen2.5-1.5B-Instruct
Loading model (4-bit) : Qwen/Qwen2.5-1.5B-Instruct


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Model loaded | VRAM used: 1.2 / 15.6 GB


## Section 5 — Configure LoRA Adapters

In [8]:
# LoRA targets the attention + MLP projection layers in Qwen2.5
lora_config = LoraConfig(
    r=16,                    # rank — higher = more capacity, more memory
    lora_alpha=32,           # scaling factor (typically 2× rank)
    target_modules=[         # Qwen2.5 attention & MLP layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## Section 6 — Fine-tune with SFTTrainer

In [9]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=False,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=5,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
    dataset_text_field="text",    # moved here from SFTTrainer
    max_length=1024,          # moved here from SFTTrainer
    packing=False,                # moved here from SFTTrainer
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
print(train_dataset[0]["text"])
print("Starting fine-tuning...")
trainer.train()
print("✓ Fine-tuning complete")


Adding EOS to train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


<|im_start|>system
You are a knowledgeable and friendly interior design consultant. Give practical, actionable advice about space planning, colour palettes, lighting, furniture arrangement, and design styles.<|im_end|>
<|im_start|>user
What are the best flooring options for a family home?<|im_end|>
<|im_start|>assistant
Luxury vinyl plank (LVP) is highly durable, waterproof, and scratch-resistant — ideal for high-traffic areas and homes with children or pets. Engineered hardwood offers the warmth of real wood with better moisture resistance than solid hardwood. Porcelain tiles work well in kitchens and bathrooms. Avoid solid hardwood in humid areas or below grade.<|im_end|>

Starting fine-tuning...


Step,Training Loss,Validation Loss
5,3.167728,2.795392
10,2.367257,2.124978


✓ Fine-tuning complete


## Section 7 — Save Fine-tuned Adapters

In [10]:
# Save only the LoRA adapter weights (small — a few MB)
ADAPTER_DIR = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✓ Adapter saved to: {ADAPTER_DIR}")

# Optional: merge LoRA weights into the base model and save as a full model
# (larger file, but easier to deploy — no PEFT library needed at inference)
#
# from peft import AutoPeftModelForCausalLM
# merged = AutoPeftModelForCausalLM.from_pretrained(ADAPTER_DIR, torch_dtype=torch.float16)
# merged = merged.merge_and_unload()
# merged.save_pretrained(f"{OUTPUT_DIR}/merged")
# print("✓ Merged model saved")

✓ Adapter saved to: ./qwen-interior-design-qlora/final-adapter


## Section 8 — Test Fine-tuned Model

In [11]:
# Load the fine-tuned model from saved adapter
from peft import AutoPeftModelForCausalLM

ft_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,   # keep 4-bit for inference too
)
ft_model.eval()
print("✓ Fine-tuned model loaded")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✓ Fine-tuned model loaded


In [12]:
def generate(user_question: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": user_question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(ft_model.device)

    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Test questions
questions = [
    "What colors work best for a small living room?",
    "How do I create a Scandinavian-style bedroom on a budget?",
    "What lighting should I use for a home office?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate(q)}")
    print("-" * 60)

Q: What colors work best for a small living room?
A: A small living room can be intimidating but it doesn't have to be! The trick is to use the right color palette. A good rule of thumb is to choose cool tones such as blues or grays that will make the space feel larger than it actually is. You could also consider using warm tones like yellows or oranges if you want something more lively.
------------------------------------------------------------
Q: How do I create a Scandinavian-style bedroom on a budget?
A: The goal of this room is to provide comfort, peace, and relaxation in the form of light and airiness. The walls should be painted in shades of soft neutral colours such as beige or cream. This will create an airy atmosphere that makes you feel comfortable when relaxing.

The flooring can also be chosen for its aesthetic appeal. A wooden floor is a good option because it provides warmth and creates a cozy environment. It's important to choose a low-slung bed frame so that your hea

## Section 9 — Interactive Chat UI

---
## Example Questions to Try
- *"What colors work well in a small living room?"*
- *"How do I arrange furniture in an open-plan space?"*
- *"What are the key principles of Scandinavian interior design?"*
- *"How can I add warmth to a room without repainting?"*
- *"What lighting should I use for a home office?"*

## Section 10 — VLM Edit Plan Analyser & Improver

In [13]:
import json

# ── Paste your VLM JSON output here ─────────────────────────────────────────
vlm_output = {
  "image_path": "uploads/365Cupperserangoonroad_11.jpg",
  "user_prompt": "I want a Japanese themed house",
  "vision_output": {
    "objects": ["desk","chair","bed","window","track lighting","shelving unit","plant"],
    "room_type": "bedroom",
    "scene_style": "modern",
    "object_notes": {
      "desk": "long wooden desk",
      "chair": "simple beige chair",
      "bed": "white bed with striped blanket",
      "window": "large window with blinds",
      "track lighting": "black track lighting fixtures",
      "shelving unit": "wooden shelving unit on wall",
      "plant": "potted plant on windowsill"
    }
  },
  "edit_plan": {
    "global_style": "Japanese themed house",
    "edits": [
      {"object": "desk",          "description": "a traditional Japanese low table with a natural wood finish"},
      {"object": "chair",         "description": "a zabuton cushion for sitting on the floor"},
      {"object": "bed",           "description": "a futon mattress with a woven cover"},
      {"object": "window",        "description": "paper shoji screens instead of blinds"},
      {"object": "track lighting","description": "paper lanterns hanging from the ceiling"},
      {"object": "shelving unit", "description": "a built-in bamboo shelf unit"},
      {"object": "plant",         "description": "a bonsai tree in a traditional pot"}
    ]
  }
}

# ── Helper: build a structured prompt from the VLM JSON ─────────────────────
def build_analysis_prompt(vlm: dict) -> str:
    vout   = vlm["vision_output"]
    eplan  = vlm["edit_plan"]
    prompt = vlm["user_prompt"]

    edit_map = {e["object"]: e["description"] for e in eplan["edits"]}
    objects_block = "\n".join(
        f"  - {obj} | currently: {desc} | proposed edit: {edit_map.get(obj, 'no change')}"
        for obj, desc in vout["object_notes"].items()
    )

    return (
        f"You are an expert interior design consultant preparing image editing prompts "
        f"for a {eplan['global_style']} redesign of a {vout['room_type']} "
        f"(currently {vout['scene_style']} style).\n"
        f"The user wants: \"{prompt}\"\n\n"
        f"Objects and their proposed edits:\n{objects_block}\n\n"
        f"For each object, write ONE sentence that an image editing model will use "
        f"to generate the replacement for that object.\n\n"
        f"Each line must start directly with the object description — no 'Replace with' prefix. "
        f"Write it as: 'A [object] in [material], [colour], [surface appearance].' "
        f"One sentence, under 25 words.\n\n"
        f"NEVER use: 'soft', 'cozy', 'elegant', 'natural wood', 'fabric', 'traditional style', "
        f"'showing', 'featuring'.\n\n"
        f"Use EXACTLY this format and nothing else:\n"
        f"- **Desk**: [one sentence]\n"
        f"- **Chair**: [one sentence]\n"
        f"- **Bed**: [one sentence]\n"
        f"- **Window**: [one sentence]\n"
        f"- **Track Lighting**: [one sentence]\n"
        f"- **Shelving Unit**: [one sentence]\n"
        f"- **Plant**: [one sentence]\n\n"
        f"Begin directly with the list. No preamble, no evaluation, no summary."
    )



# ── Run the fine-tuned model on the VLM JSON ─────────────────────────────────
analysis_prompt = build_analysis_prompt(vlm_output)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": analysis_prompt},
]
text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(ft_model.device)

with torch.no_grad():
    output_ids = ft_model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.75,
        top_p=0.92,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )


new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
improved   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("=" * 60)
print(f"User prompt : {vlm_output['user_prompt']}")
print(f"Room type   : {vlm_output['vision_output']['room_type']}")
print("=" * 60)
print("\n📋 IMPROVED EDIT PLAN:\n")
print(improved)

User prompt : I want a Japanese themed house
Room type   : bedroom

📋 IMPROVED EDIT PLAN:

Desk: A traditional Japanese low table with a smooth, glossy surface.

Chair: A zafu cushion made from a thick, soft fabric.

Bed: A futon mattress wrapped in a richly textured, luxurious fabric.

Window: Paper shoji screens installed with clear, lightweight materials.

Track Lighting: Hanging paper lanterns hung at various heights using cord suspensions.

Shelving Unit: A bookshelf covered with dark, deep brown leather.

Plant: A small, intricately carved bonsai tree placed in a shallow dish.


In [14]:
import time, re, statistics, json
from datetime import datetime
from pathlib import Path

# ── 1. Base room (same for all styles) ───────────────────────────────────────
BASE_ROOM = {
    "objects": ["desk","chair","bed","window","track lighting","shelving unit","plant"],
    "room_type": "bedroom",
    "scene_style": "modern",
    "object_notes": {
        "desk":           "long wooden desk",
        "chair":          "simple beige chair",
        "bed":            "white bed with striped blanket",
        "window":         "large window with blinds",
        "track lighting": "black track lighting fixtures",
        "shelving unit":  "wooden shelving unit on wall",
        "plant":          "potted plant on windowsill",
    }
}

# ── 2. One edit plan per style ────────────────────────────────────────────────
DESIGNS = {
    "japanese": {
        "prompt": "I want a Japanese themed bedroom",
        "edits": [
            {"object": "desk",           "description": "a traditional low chabudai table with a natural wood finish"},
            {"object": "chair",          "description": "a zabuton floor cushion for seated work"},
            {"object": "bed",            "description": "a futon mattress with a woven indigo cover"},
            {"object": "window",         "description": "paper shoji screens replacing the blinds"},
            {"object": "track lighting", "description": "paper lanterns suspended from the ceiling"},
            {"object": "shelving unit",  "description": "a built-in bamboo shelf unit"},
            {"object": "plant",          "description": "a bonsai tree in a glazed ceramic pot"},
        ]
    },
    "victorian": {
        "prompt": "I want a Victorian style bedroom",
        "edits": [
            {"object": "desk",           "description": "an ornate writing desk with carved dark mahogany legs"},
            {"object": "chair",          "description": "a tufted wingback chair in deep burgundy velvet"},
            {"object": "bed",            "description": "a four-poster bed with heavy damask drapes"},
            {"object": "window",         "description": "floor-length velvet curtains with brass tiebacks"},
            {"object": "track lighting", "description": "a brass chandelier with frosted glass globe shades"},
            {"object": "shelving unit",  "description": "a dark walnut bookcase with glass doors and brass hardware"},
            {"object": "plant",          "description": "a large fern in a brass jardinière stand"},
        ]
    },
    "modern": {
        "prompt": "I want a modern minimalist bedroom",
        "edits": [
            {"object": "desk",           "description": "a wall-mounted floating desk in white lacquer"},
            {"object": "chair",          "description": "a moulded shell chair on slim hairpin legs"},
            {"object": "bed",            "description": "a low platform bed with an upholstered light grey headboard"},
            {"object": "window",         "description": "floor-to-ceiling roller blinds in warm white"},
            {"object": "track lighting", "description": "recessed LED spotlights flush to the ceiling"},
            {"object": "shelving unit",  "description": "a modular open shelving system in white powder-coated steel"},
            {"object": "plant",          "description": "a single sculptural cactus in a concrete planter"},
        ]
    },
    "scandinavian": {
        "prompt": "I want a Scandinavian hygge bedroom",
        "edits": [
            {"object": "desk",           "description": "a light birch desk with tapered solid wood legs"},
            {"object": "chair",          "description": "a sheepskin-draped solid ash stool"},
            {"object": "bed",            "description": "a low ash frame with a white cotton duvet and knit throw"},
            {"object": "window",         "description": "sheer undyed linen curtains that pool softly on the floor"},
            {"object": "track lighting", "description": "a white paper pendant lamp on a fabric cord"},
            {"object": "shelving unit",  "description": "a floating light pine shelf with minimal objects"},
            {"object": "plant",          "description": "eucalyptus stems in a simple matte ceramic vase"},
        ]
    },
    "industrial": {
        "prompt": "I want an industrial loft bedroom",
        "edits": [
            {"object": "desk",           "description": "a reclaimed oak plank desk on black iron pipe legs"},
            {"object": "chair",          "description": "a metal factory stool with a worn leather seat pad"},
            {"object": "bed",            "description": "a black powder-coated steel bed frame with visible hex bolts"},
            {"object": "window",         "description": "steel-framed windows left bare to maximise light"},
            {"object": "track lighting", "description": "bare Edison bulb pendants on black iron conduit pipe"},
            {"object": "shelving unit",  "description": "iron pipe brackets holding raw-edge reclaimed wood planks"},
            {"object": "plant",          "description": "a tall snake plant in a galvanised steel bucket"},
        ]
    },
    "bohemian": {
        "prompt": "I want a bohemian eclectic bedroom",
        "edits": [
            {"object": "desk",           "description": "a distressed pine desk with a macrame-trimmed drawer pull"},
            {"object": "chair",          "description": "a rattan hanging egg chair with layered kilim cushions"},
            {"object": "bed",            "description": "a low divan layered with patterned textiles and throw pillows"},
            {"object": "window",         "description": "sheer embroidered muslin curtains with tassel trim"},
            {"object": "track lighting", "description": "warm string fairy lights and a woven rattan pendant"},
            {"object": "shelving unit",  "description": "mismatched reclaimed wood floating shelves with trailing plants"},
            {"object": "plant",          "description": "trailing pothos in a hanging macrame plant holder"},
        ]
    },
}

# ── 3. Scoring helpers ────────────────────────────────────────────────────────
GENERIC_TERMS = [
    "natural wood", "soft fabric", "traditional style", "traditional material",
    "wooden material", "nice", "elegant", "classic",
]
SPECIFIC_PATTERNS = [
    r"\b(hinoki|sugi|cedar|cypress|paulownia|kiri|persimmon|mahogany|walnut|birch|ash|oak|pine|teak)\b",
    r"\b(ramie|habotai|asa|hemp|linen|washi|kozo|velvet|damask|muslin|kilim|cotton|sheepskin)\b",
    r"\b(urushi|lacquer|tung.oil|beeswax|shou.sugi|charred|powder.coated|galvanised)\b",
    r"\b(shigaraki|bizen|stoneware|brass|iron|steel)\b",
    r"\d+\s*(cm|mm|inches?)",
    r"#[0-9a-fA-F]{6}",
]

def score_completeness(output: str, vlm: dict) -> dict:
    objects = [e["object"].lower() for e in vlm["edit_plan"]["edits"]]
    covered = [o for o in objects if o in output.lower()]
    return {
        "score": round(len(covered) / len(objects), 3),
        "covered": covered,
        "missing": [o for o in objects if o not in covered],
    }

def score_specificity(output: str) -> dict:
    text = output.lower()
    generic_hits = sum(1 for t in GENERIC_TERMS if t in text)
    specific_hits, found = 0, []
    for pat in SPECIFIC_PATTERNS:
        matches = re.findall(pat, text)
        specific_hits += len(matches)
        found.extend(matches)
    total = specific_hits + generic_hits
    return {
        "score": round(specific_hits / total, 3) if total > 0 else 0.0,
        "specific_count": specific_hits,
        "generic_count": generic_hits,
        "found": list(set(found))[:8],
    }

def run_once(vlm: dict) -> tuple:
    prompt = build_analysis_prompt(vlm)
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": prompt}]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(ft_model.device)
    t0 = time.time()
    with torch.no_grad():
        ids = ft_model.generate(
            **inputs, max_new_tokens=1000, temperature=0.75,
            top_p=0.92, do_sample=True, repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    new_tokens = ids[0][inputs.input_ids.shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded, elapsed, len(new_tokens)

# ── 4. Run benchmark ──────────────────────────────────────────────────────────
N_RUNS   = 3
all_results = {}

for style, cfg in DESIGNS.items():
    vlm = {
        "image_path":    "uploads/test_room.jpg",
        "user_prompt":   cfg["prompt"],
        "vision_output": BASE_ROOM,
        "edit_plan":     {"global_style": f"{style} style", "edits": cfg["edits"]},
    }

    print(f"\n{'='*58}")
    print(f"  {style.upper():<20} ({N_RUNS} runs)")
    print(f"{'='*58}")

    runs = []
    for i in range(N_RUNS):
        print(f"  run {i+1}/{N_RUNS} ...", end=" ", flush=True)
        out, elapsed, n_tok = run_once(vlm)
        c = score_completeness(out, vlm)
        s = score_specificity(out)
        print(f"{elapsed:.1f}s | comp={c['score']*100:.0f}% | spec={s['score']*100:.0f}%")
        runs.append({
            "run":          i + 1,
            "output":       out,
            "completeness": c,
            "specificity":  s,
            "speed":        {"elapsed_s": round(elapsed, 2), "tokens": n_tok,
                             "tok_per_s": round(n_tok / elapsed, 1)},
        })

    c_vals = [r["completeness"]["score"] for r in runs]
    s_vals = [r["specificity"]["score"]  for r in runs]
    t_vals = [r["speed"]["elapsed_s"]    for r in runs]

    all_results[style] = {
        "runs": runs,
        "summary": {
            "completeness_mean":  round(statistics.mean(c_vals), 3),
            "completeness_stdev": round(statistics.stdev(c_vals), 3),
            "specificity_mean":   round(statistics.mean(s_vals), 3),
            "specificity_stdev":  round(statistics.stdev(s_vals), 3),
            "speed_mean_s":       round(statistics.mean(t_vals), 2),
            "speed_stdev_s":      round(statistics.stdev(t_vals), 2),
        }
    }

# ── 5. Save JSON ──────────────────────────────────────────────────────────────
out_path = Path("benchmark_results.json")
out_path.write_text(json.dumps({
    "timestamp":        datetime.now().isoformat(),
    "model":            MODEL_ID,
    "n_runs_per_style": N_RUNS,
    "results":          all_results,
}, indent=2))
print(f"\n✓ Full results saved → {out_path}")

# ── 6. Save markdown report ───────────────────────────────────────────────────
md_lines = [
    f"# Benchmark Report\n",
    f"**Model:** {MODEL_ID}  |  **Runs per style:** {N_RUNS}  |  "
    f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n",
    "| Style | Comp mean | Comp ± | Spec mean | Spec ± | Speed (s) | Speed ± |",
    "|---|---|---|---|---|---|---|",
]
for style, data in all_results.items():
    s = data["summary"]
    md_lines.append(
        f"| {style} | {s['completeness_mean']:.2f} | {s['completeness_stdev']:.3f} "
        f"| {s['specificity_mean']:.2f} | {s['specificity_stdev']:.3f} "
        f"| {s['speed_mean_s']:.1f} | {s['speed_stdev_s']:.2f} |"
    )

md_lines += ["\n## Outputs\n"]
for style, data in all_results.items():
    md_lines.append(f"### {style.title()}")
    for r in data["runs"]:
        md_lines.append(f"\n**Run {r['run']}** — "
                        f"comp={r['completeness']['score']*100:.0f}% | "
                        f"spec={r['specificity']['score']*100:.0f}% | "
                        f"{r['speed']['elapsed_s']}s\n")
        md_lines.append(r["output"])
        md_lines.append("\n---")

Path("benchmark_report.md").write_text("\n".join(md_lines))
print("✓ Markdown report saved → benchmark_report.md")

# ── 7. Print summary table ────────────────────────────────────────────────────
print(f"\n{'Style':<15} {'Comp':>6} {'±':>6} {'Spec':>6} {'±':>6} {'Speed':>8} {'±':>6}")
print("-" * 62)
for style, data in all_results.items():
    s = data["summary"]
    print(f"{style:<15} {s['completeness_mean']:>6.2f} {s['completeness_stdev']:>6.3f} "
          f"{s['specificity_mean']:>6.2f} {s['specificity_stdev']:>6.3f} "
          f"{s['speed_mean_s']:>8.1f} {s['speed_stdev_s']:>6.2f}")



  JAPANESE             (3 runs)
  run 1/3 ... 20.8s | comp=100% | spec=0%
  run 2/3 ... 32.8s | comp=100% | spec=100%
  run 3/3 ... 22.4s | comp=100% | spec=100%

  VICTORIAN            (3 runs)
  run 1/3 ... 14.7s | comp=100% | spec=100%
  run 2/3 ... 25.9s | comp=100% | spec=100%
  run 3/3 ... 15.9s | comp=100% | spec=100%

  MODERN               (3 runs)
  run 1/3 ... 20.5s | comp=100% | spec=75%
  run 2/3 ... 27.2s | comp=100% | spec=100%
  run 3/3 ... 18.5s | comp=100% | spec=100%

  SCANDINAVIAN         (3 runs)
  run 1/3 ... 27.0s | comp=100% | spec=88%
  run 2/3 ... 20.9s | comp=100% | spec=100%
  run 3/3 ... 20.9s | comp=100% | spec=75%

  INDUSTRIAL           (3 runs)
  run 1/3 ... 22.2s | comp=100% | spec=100%
  run 2/3 ... 20.8s | comp=100% | spec=100%
  run 3/3 ... 19.2s | comp=100% | spec=100%

  BOHEMIAN             (3 runs)
  run 1/3 ... 14.9s | comp=100% | spec=100%
  run 2/3 ... 34.2s | comp=100% | spec=100%
  run 3/3 ... 15.0s | comp=100% | spec=100%

✓ Full results